# 3. 系统增强

这一节关注"系统级"问题：当单次检索流程已经优化过，仍需处理多轮记忆、多文档路由、工具编排与可恢复执行时，如何构建更稳健的 RAG 系统。

## 本章要解决什么

前两章解决的是单次请求内的问题——上下文不全、流程不够。但当系统需要记住上一轮对话、协调多个数据源、或在复杂任务中自主规划执行路径时，单次请求的优化已经不够了。

本章关注的是**跨请求的工程问题**：状态怎么延续、工具怎么组织、复杂任务怎么规划与执行。我们会从最基础的记忆管理开始，逐步引入多文档路由、知识图谱，最后用 Agentic RAG 把这些能力组装成一个完整的自主系统。

## 流程增强 vs 系统增强

流程增强关注"单次请求内"的多步决策（多轮检索、子问题拆解、质量把关）。

系统增强关注"跨请求"的工程问题：
- 上一轮对话的信息怎么延续？-> Memory
- 多个数据源怎么组织和路由？-> Multi-Document Agent
- 复杂任务怎么规划、执行、反思？-> Agentic RAG

判断标准：如果去掉 history / 去掉多文档路由 / 去掉任务规划，系统仍能回答，那是流程问题；否则是系统问题。


## 统一实验设置

与前两节保持一致：

- **数据**：`../3. 索引阶段/data/pumpkin_book.pdf`（南瓜书《机器学习公式详解》）
- **问答数据**：`../3. 索引阶段/data/train_dataset.json`（选取前 5 个问答对用于实验）
- **生成模型**：`glm-4-flash-250414`
- **向量模型**：本地 `BAAI/bge-small-zh-v1.5`
- **评估**：使用 LLM 作为裁判进行评估

## 环境准备

本节使用智谱 AI 的 `GLM-4-Flash` 做生成模型，使用本地 `BAAI/bge-small-zh-v1.5` 做 embedding。运行前请确保：

1. 安装依赖：`pip install langchain langchain-community langchain-chroma zhipuai python-dotenv pymupdf pandas modelscope sentence-transformers transformers torch`
2. 在项目根目录的 `.env` 文件中配置 `ZHIPUAI_API_KEY`
3. 首次运行会自动从 ModelScope 下载本地 embedding 模型到当前目录下的 `./models/`


In [1]:
import os
import re
import json
import warnings
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import sys
sys.path.insert(0, ".")
from _common import (
    get_embeddings, get_cleaned_pdf_documents, open_or_build_chroma,
    llm_call as _raw_llm_call, build_rag_generation_prompt,
    trim_context_to_budget, simple_eval_2pt, build_compare_table,
    CHROMA_COLLECTION, CONTEXT_CHAR_BUDGET, PDF_PATH, QA_PATH,
)

warnings.filterwarnings("ignore")


def llm_call(prompt: str) -> str:
    """6.3 节默认每次成功调用后 sleep 1 秒。"""
    return _raw_llm_call(prompt, sleep_after=1.0)


def load_chunks(chunk_size=256, chunk_overlap=20):
    docs = list(get_cleaned_pdf_documents())
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""], keep_separator=True,
    )
    return splitter.split_documents(docs)


def build_retriever(chunk_size=256, chunk_overlap=20, k=4, persist_subdir="baseline_256_20"):
    persist_dir = f"./chroma_db/{persist_subdir}"
    chunks = load_chunks(chunk_size, chunk_overlap)
    ids = [f"b{i}" for i in range(len(chunks))]
    vs = open_or_build_chroma(persist_dir, chunks, ids)
    return vs.as_retriever(search_kwargs={"k": k})


retriever = build_retriever()
print("✅ 6.3 环境准备完成（公共底座来自 _common）")


/usr/local/Caskroom/miniconda/base/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  -> 加载已有索引: ./chroma_db/baseline_256_20
✅ 6.3 环境准备完成（公共底座来自 _common）


## 本节钩子：`class XxxSystem: ask(q) -> str`

6.1 钩子是 `*_context`（变 context 拼法），6.2 钩子是 `*_pipeline`（变流程编排）。
但系统增强的核心是 **状态**——跨轮记忆、跨源路由——单题接口 `q -> str` 根本
表达不出来。所以本节钩子升一层：每个方法是一个 **带状态的类**，对外提供 `ask(q)`。

```python
class MemoryRAGSystem:    def ask(q) -> str   # 内部更新 history
class MultiDocAgent:      def ask(q) -> str   # 内部路由到不同 source
```

公共底座为此提供 `run_session_eval(system, qna_dict)`：按顺序调用 `system.ask`，
**复用同一 system 实例** 以保留状态。它是 6.3 首次出现的概念，下面会先 inline
完整定义一次，再放进 `_common.py` 供未来扩展使用。


## Memory：多轮对话 RAG

多轮对话最大的麻烦是 **指代**——"它的候选划分点是怎么确定的？" 这里的"它"指什么？
单看一句话，retriever 完全无法定位。`MemoryRAGSystem` 的做法是：每次 `ask(q)` 时，
先用 LLM 把 `(history, q)` 改写成一个 **不依赖历史** 的独立查询（condensed query），
再用这个独立查询去 retrieve + generate，最后把 `(原始 q, 答案)` 入 history。

这种"先改写、再检索"的模式在工业实现中被称为 **Condense Question Chain**。

In [2]:
def run_session_eval(system, qna_dict, *, eval_prompt_template=None):
    """按顺序调 system.ask(q)，复用同一 system 实例以保留状态。"""
    rows = []
    for question, expected in qna_dict.items():
        answer = system.ask(question)
        score = simple_eval_2pt(
            answer, expected, question, prompt_template=eval_prompt_template,
        )
        rows.append({
            "question": question, "llm_answer": answer,
            "expected_answer": expected, "rag_eval_results": score,
        })
    return pd.DataFrame(rows)


MEMORY_BEHAVIORAL_EVAL_PROMPT = (
    "你是判卷人。本题是多轮对话中的一轮，「用户问题」可能含指代（如『它』、"
    "『上面提到的方法』）。请按 0~2 分评估「模型答案」：\n"
    "2 分：正确解析指代，并给出与「参考答案」核心一致的回答。\n"
    "1 分：解析了指代但答案部分缺失或方向正确但不完整。\n"
    "0 分：未解析指代、答非所问、或与参考答案明显矛盾。\n\n"
    "用户问题：{question}\n参考答案：{expected_answer}\n模型答案：{llm_answer}\n\n"
    "仅输出一行，只包含字符 0、1 或 2。"
)


> 上面 `run_session_eval` 已收纳到 `_common.py`，本节后续与未来章节通过 `import` 复用。


In [3]:
class MemoryRAGSystem:
    """多轮对话 RAG：每次 ask 时，先用 LLM 把 (history, q) 改写成独立 query，
    再正常 retrieve + generate；新 (q, a) 入 history。"""

    def __init__(self, retriever, max_history: int = 5):
        self.retriever = retriever
        self.history: list[tuple[str, str]] = []
        self.max_history = max_history
        self.last_debug: dict = {}

    def _condense(self, q: str) -> str:
        if not self.history:
            return q
        hist_text = "\n".join(f"Q: {hq}\nA: {ha}" for hq, ha in self.history)
        prompt = (
            "下面是历史对话。请把最新的问题改写成一个不依赖历史的独立问题，"
            "保留所有指代消解后的实体名。只输出改写后的问题，不要任何前缀。\n\n"
            f"历史：\n{hist_text}\n\n最新问题：{q}\n\n改写："
        )
        return llm_call(prompt).strip()

    def ask(self, q: str) -> str:
        condensed = self._condense(q)
        docs = self.retriever.invoke(condensed)
        ctx = trim_context_to_budget(
            "\n\n".join(d.page_content for d in docs),
            CONTEXT_CHAR_BUDGET,
        )
        ans = llm_call(build_rag_generation_prompt(q, ctx))
        self.last_debug = {
            "original": q, "condensed": condensed,
            "n_hits": len(docs),
            "first_hit_head": docs[0].page_content[:80] if docs else "",
        }
        self.history.append((q, ans))
        self.history = self.history[-self.max_history :]
        return ans


### 单 session 观察：condensed 改写发生了吗？

先只跑第 1 个 session（决策树连续属性），打印每轮的 `condensed` 与命中变化。
我们关心的客观信号是：从 turn 2 开始，`condensed` 应该不等于原问题
（因为原问题"它的..."独立看是无意义的，必须被改写）。

In [4]:
def inspect_memory_session(session: dict) -> None:
    print(f"━━━━━ Session: {session['title']} ━━━━━\n")
    sys = MemoryRAGSystem(retriever)
    for i, turn in enumerate(session["turns"]):
        q = turn["q"]
        ans = sys.ask(q)
        d = sys.last_debug
        print(f"--- Turn {i+1} ---")
        print(f"  原问题：{q}")
        print(f"  Condensed：{d['condensed']}")
        print(f"  改写发生：{'是' if d['condensed'] != q else '否'}")
        print(f"  命中数：{d['n_hits']}；首条命中：{d['first_hit_head']}...")
        print(f"  答案：{ans[:160]}...")
        print()


with open("data/memory_sessions.json", "r", encoding="utf-8") as f:
    SESSIONS = json.load(f)["sessions"]

inspect_memory_session(SESSIONS[0])


━━━━━ Session: 决策树连续属性 ━━━━━



--- Turn 1 ---
  原问题：决策树在处理连续属性时常用什么离散化方法？
  Condensed：决策树在处理连续属性时常用什么离散化方法？
  改写发生：否
  命中数：4；首条命中：算法、“嵌入式”算法的概念，若先使用某个离散化算法对连续属性离散化后再调用C4.5决策树生成算法，则是一种过滤式算法，若如4.4.1节所述，则应该属于嵌入式算法...
  答案：上下文中没有提到决策树在处理连续属性时常用的离散化方法。...



--- Turn 2 ---
  原问题：它的候选划分点是怎么确定的？
  Condensed：决策树的候选划分点是怎么确定的？
  改写发生：是
  命中数：4；首条命中：第4章决策树本章的决策树算法背后没有复杂的数学推导，其更符合人类日常思维方式，理解起来也更为直观，其引入的数学工具也仅是为了让该算法在计算上可行，同时“西瓜书”...
  答案：根据上下文，CART决策树的候选划分点的确定方法如下：

1.  考虑每个属性a的每个可能取值v，将数据集D分为a=v和a̸=v两部分来计算基尼指数，即Gini_index(D,a)=|Da=v||D|Gini(Da=v)+|Da̸=v||D|Gini(Da̸=v)。
2.  选择基尼指数最小的属性及其对应取值作为最优...



--- Turn 3 ---
  原问题：选择最优划分点的准则是什么？
  Condensed：选择最优划分点的准则是什么？
  改写发生：否
  命中数：4；首条命中：优划分点的计算过程如下：以属性“色泽”为例，它有3个可能的取值：{青绿，乌黑，浅白}，若使用该属性的属性值是否等于“青绿”对数据集D进行划分，则可得到2个子集，...
  答案：选择最优划分点的准则是选择基尼指数最小的属性及其对应取值作为最优划分属性和最优划分点。...



### 行为评估：memory vs no-memory baseline

在 inspect 之外，还需要一个量化对比。对照组 `NoMemorySystem` 不维护 history，
每题独立检索；它在 turn 2/3 上 **应当** 因为读不懂指代而失分。我们用
`MEMORY_BEHAVIORAL_EVAL_PROMPT`（针对指代解析的 0~2 分判卷）跑全部 3 个 session，
合并后用 `build_compare_table` 出对比表。

In [5]:
def session_to_qna(session):
    return {turn["q"]: turn["a"] for turn in session["turns"]}


def baseline_no_memory_ask_factory():
    """对照组：每题独立 retrieve + generate，无 history。"""
    class NoMemorySystem:
        def __init__(self, retriever):
            self.retriever = retriever
        def ask(self, q):
            docs = self.retriever.invoke(q)
            ctx = trim_context_to_budget(
                "\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET,
            )
            return llm_call(build_rag_generation_prompt(q, ctx))
    return NoMemorySystem(retriever)


memory_dfs, baseline_dfs = [], []
for sess in SESSIONS:
    qna = session_to_qna(sess)
    mem_sys = MemoryRAGSystem(retriever)
    base_sys = baseline_no_memory_ask_factory()
    memory_dfs.append(run_session_eval(mem_sys, qna, eval_prompt_template=MEMORY_BEHAVIORAL_EVAL_PROMPT))
    baseline_dfs.append(run_session_eval(base_sys, qna, eval_prompt_template=MEMORY_BEHAVIORAL_EVAL_PROMPT))

memory_all = pd.concat(memory_dfs, ignore_index=True)
baseline_all = pd.concat(baseline_dfs, ignore_index=True)
memory_compare = build_compare_table(
    [baseline_all, memory_all],
    names=["no_memory_baseline", "memory"],
)
memory_compare


,question,no_memory_baseline,memory
0,F1 是怎么把它们组合起来的？,0,1
1,上面提到的方法各自适合什么数据规模？,0,0
2,什么是查准率（precision）和查全率（recall）？,2,2
3,其中哪种会改变训练集的样本分布？,0,0
4,决策树在处理连续属性时常用什么离散化方法？,0,0
5,如果想偏重其中之一，可以用什么变体？,0,1
6,它的候选划分点是怎么确定的？,0,1
7,机器学习中常用的模型评估方法有哪些？,0,0
8,选择最优划分点的准则是什么？,1,1


### Memory 表怎么读

- `no_memory_baseline` 列：在 turn 2/3 上低分意味着模型把"它"、"上面提到"等指代
  当成实体来检索，召回到无关章节或答非所问。
- `memory` 列：高分需要两个条件同时满足——condensed 改写正确 + 改写后的查询能命中。
- 如果两列都偏低，往往是 retriever k 不够或 chunk 切得太碎，需要回到 6.1/6.2 调；
  Memory 解决的是 **指代消解**，不是检索召回的根本问题。

## Multi-Document Agent（多源路由）

Memory 解决"跨轮"，Multi-Doc Agent 解决"跨源"。当文档分布在多个领域 / 章节时，
盲检全库会把无关章节的相似片段也召回。Multi-Doc Agent 的核心是：
**每个 source 一个独立的检索器 + 一个 LLM 路由器**——根据问题选 1~3 个 source，
仅在选中的 source 上检索。

下面把南瓜书按页区间切成若干"章节 source"，建独立的 Chroma 集合。

In [6]:
def split_by_page_ranges(page_ranges: dict[str, tuple[int, int]]):
    """page_ranges 例：{'ch1_intro': (0, 18), 'ch2_eval': (19, 45), ...}
    返回 {source_name: list[Document]}。
    """
    pages = get_cleaned_pdf_documents()
    out: dict[str, list[Document]] = {}
    for name, (lo, hi) in page_ranges.items():
        docs = []
        for p in pages:
            page_idx = p.metadata.get("page", -1)
            if lo <= page_idx <= hi and p.page_content.strip():
                docs.append(p)
        out[name] = docs
    return out


SOURCE_PAGE_RANGES = {
    "ch1_intro":     (0, 18),
    "ch2_eval":      (19, 45),
    "ch3_linear":    (46, 75),
    "ch4_tree":      (76, 110),
}

SOURCE_DESCRIPTIONS = {
    "ch1_intro":     "第1章 绪论：机器学习基本术语、假设空间、归纳偏好。",
    "ch2_eval":      "第2章 模型评估与选择：留出法、交叉验证、自助法、查准查全率、ROC、偏差方差。",
    "ch3_linear":    "第3章 线性模型：线性回归、对数几率回归、LDA、多分类、类别不平衡。",
    "ch4_tree":      "第4章 决策树：信息增益、增益率、基尼指数、剪枝、连续与缺失值。",
}


def build_chapter_sources():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=256, chunk_overlap=20,
        separators=["\n\n", "\n", " ", ""], keep_separator=True,
    )
    by_source = split_by_page_ranges(SOURCE_PAGE_RANGES)
    retrievers = {}
    for name, page_docs in by_source.items():
        chunks = splitter.split_documents(page_docs)
        if not chunks:
            print(f"  [warn] source {name} 切完为空，跳过")
            continue
        ids = [f"{name}-{i}" for i in range(len(chunks))]
        vs = open_or_build_chroma(f"./chroma_db/multi_doc/{name}", chunks, ids)
        retrievers[name] = vs.as_retriever(search_kwargs={"k": 4})
    return retrievers


CHAPTER_RETRIEVERS = build_chapter_sources()
print(f"✅ 多源构建完成，可用 sources：{list(CHAPTER_RETRIEVERS.keys())}")


  -> 创建新索引: ./chroma_db/multi_doc/ch1_intro


  -> 创建新索引: ./chroma_db/multi_doc/ch2_eval


  -> 创建新索引: ./chroma_db/multi_doc/ch3_linear


  -> 创建新索引: ./chroma_db/multi_doc/ch4_tree


✅ 多源构建完成，可用 sources：['ch1_intro', 'ch2_eval', 'ch3_linear', 'ch4_tree']


In [7]:
class MultiDocAgent:
    """多源路由 RAG：LLM 根据问题与 source 描述选 1~3 个 source，仅在选中源检索。"""

    def __init__(self, sources: dict, descriptions: dict):
        self.sources = sources
        self.descriptions = descriptions
        self.last_debug: dict = {}

    def _route(self, q: str) -> list[str]:
        desc_text = "\n".join(f"- {k}: {v}" for k, v in self.descriptions.items())
        prompt = (
            "你是一个路由器。给定可选的知识源描述与用户问题，"
            "请输出 1~3 个最相关的 source key（每行一个 key，不要任何其它内容）。\n\n"
            f"可选源：\n{desc_text}\n\n问题：{q}\n\n选择："
        )
        out = llm_call(prompt).strip().splitlines()
        valid = [k.strip() for k in out if k.strip() in self.sources][:3]
        return valid or [next(iter(self.sources))]

    def ask(self, q: str) -> str:
        chosen = self._route(q)
        docs = []
        per_source_hits = {}
        for name in chosen:
            r = self.sources[name].invoke(q)
            docs.extend(r)
            per_source_hits[name] = len(r)
        ctx = trim_context_to_budget(
            "\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET,
        )
        ans = llm_call(build_rag_generation_prompt(q, ctx))
        self.last_debug = {"chosen": chosen, "per_source_hits": per_source_hits}
        return ans


### 单题观察：路由器选了哪个 source？

先用两个明显归属不同章节的问题（决策树 vs 评估方法）观察路由器决策与每源命中。
合理结果：决策树类问题应主要选 `ch4_tree`；评估方法类应主要选 `ch2_eval`。

In [8]:
def inspect_multi_doc(question: str, agent: "MultiDocAgent") -> None:
    print(f"❓ 问题：{question}")
    ans = agent.ask(question)
    print(f"🧭 路由选择：{agent.last_debug['chosen']}")
    print(f"📊 各源命中：{agent.last_debug['per_source_hits']}")
    print(f"💬 答案：{ans[:200]}...\n")


agent = MultiDocAgent(CHAPTER_RETRIEVERS, SOURCE_DESCRIPTIONS)
inspect_multi_doc("决策树连续属性的二分法", agent)
inspect_multi_doc("交叉验证与留出法的区别", agent)


❓ 问题：决策树连续属性的二分法


🧭 路由选择：['ch4_tree']
📊 各源命中：{'ch4_tree': 4}
💬 答案：上下文中没有包含关于决策树连续属性二分法的完整信息。...

❓ 问题：交叉验证与留出法的区别


🧭 路由选择：['ch2_eval']
📊 各源命中：{'ch2_eval': 4}
💬 答案：上下文没有包含关于交叉验证与留出法的区别的信息。...



### 评估：全库 baseline vs 多源路由

挑 6 道在题集中、且**章节归属明确**的题，标注期望路由的 source（用于路由命中正确率）。
对照组 `FullCorpusSystem` 用一开始建好的全库 retriever；实验组是 `MultiDocAgent`。
评分采用维度风格的 0~2 分判卷（`MULTI_DOC_EVAL_PROMPT`）。

In [9]:
MULTI_DOC_EVAL_PROMPT = (
    "你是判卷人。本题需要从特定章节的知识回答。请按 0~2 分评估「模型答案」：\n"
    "2 分：与「参考答案」核心一致，无关键事实错误。\n"
    "1 分：方向正确但部分要点缺失或表述含糊。\n"
    "0 分：错误回答或与参考答案明显矛盾。\n\n"
    "用户问题：{question}\n参考答案：{expected_answer}\n模型答案：{llm_answer}\n\n"
    "仅输出一行，只包含字符 0、1 或 2。"
)


MULTI_DOC_QA = {
    0:  "ch4_tree",
    5:  "ch2_eval",
    6:  "ch2_eval",
    7:  "ch2_eval",
    26: "ch4_tree",
    27: "ch4_tree",
}

with open(QA_PATH, "r", encoding="utf-8") as f:
    _all_pairs = json.load(f)
multi_doc_qna = {_all_pairs[i]["query"]: _all_pairs[i]["answer"] for i in MULTI_DOC_QA.keys()}
expected_source_by_question = {_all_pairs[i]["query"]: src for i, src in MULTI_DOC_QA.items()}


class FullCorpusSystem:
    """对照组：单一全库 retriever，无路由。"""
    def __init__(self, retriever):
        self.retriever = retriever
    def ask(self, q):
        docs = self.retriever.invoke(q)
        ctx = trim_context_to_budget(
            "\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET,
        )
        return llm_call(build_rag_generation_prompt(q, ctx))


full_baseline = FullCorpusSystem(retriever)
agent_for_eval = MultiDocAgent(CHAPTER_RETRIEVERS, SOURCE_DESCRIPTIONS)

baseline_md_df = run_session_eval(full_baseline, multi_doc_qna, eval_prompt_template=MULTI_DOC_EVAL_PROMPT)
agent_md_df = run_session_eval(agent_for_eval, multi_doc_qna, eval_prompt_template=MULTI_DOC_EVAL_PROMPT)

multi_doc_compare = build_compare_table(
    [baseline_md_df, agent_md_df], names=["full_corpus", "multi_doc_agent"],
)
multi_doc_compare


,question,full_corpus,multi_doc_agent
0,在给定的文本中，提到了F1分数是查准率和查全率的调和平均。请解释为什么在F1分数的计算中，使...,1,1
1,根据所提供的上下文信息，请解释式(4.8)中λ的取值分别代表什么含义，并给出一个具体的例子说...,1,0
2,解释“宏平均”（macro-average）和“微平均”（micro-average）在多类...,1,1
3,解释什么是交叉验证法，并说明为什么它在评估算法性能时比单次留出法更可靠。,1,0
4,请根据图4-2中的划分过程，描述决策树是如何通过四次划分来确定好瓜与坏瓜的分类边界的。,1,0
5,请根据提供的上下文信息，解释“算法”和“模型”的概念，并说明它们在机器学习中的关系。,1,1


### 路由命中正确率（更直接的客观信号）

对比表反映"答得好不好"，但路由器到底选对没？另起一遍 ask 单看 `last_debug["chosen"]`，
统计期望 source 是否在选中集合里。验收基线：≥ 4/6 命中。如果太低，回头去
`SOURCE_DESCRIPTIONS` 把章节摘要写得更"判别性"。

In [10]:
agent_for_check = MultiDocAgent(CHAPTER_RETRIEVERS, SOURCE_DESCRIPTIONS)
correct_routes = 0
total = len(multi_doc_qna)
for q in multi_doc_qna:
    _ = agent_for_check.ask(q)
    chosen = agent_for_check.last_debug["chosen"]
    expected_src = expected_source_by_question[q]
    hit = expected_src in chosen
    correct_routes += int(hit)
    print(f"  Q={q[:40]!s}...  chosen={chosen}  expected={expected_src}  {'✅' if hit else '❌'}")
print(f"\n路由命中正确率：{correct_routes}/{total} = {correct_routes/total:.0%}")


  Q=请根据提供的上下文信息，解释“算法”和“模型”的概念，并说明它们在机器学习中的关...  chosen=['ch1_intro']  expected=ch4_tree  ❌


  Q=解释什么是交叉验证法，并说明为什么它在评估算法性能时比单次留出法更可靠。...  chosen=['ch2_eval']  expected=ch2_eval  ✅


  Q=在给定的文本中，提到了F1分数是查准率和查全率的调和平均。请解释为什么在F1分数...  chosen=['ch2_eval']  expected=ch2_eval  ✅


  Q=解释“宏平均”（macro-average）和“微平均”（micro-avera...  chosen=['ch2_eval']  expected=ch2_eval  ✅


  Q=根据所提供的上下文信息，请解释式(4.8)中λ的取值分别代表什么含义，并给出一个...  chosen=['ch4_tree']  expected=ch4_tree  ✅


  Q=请根据图4-2中的划分过程，描述决策树是如何通过四次划分来确定好瓜与坏瓜的分类边...  chosen=['ch4_tree']  expected=ch4_tree  ✅

路由命中正确率：5/6 = 83%


## 系统增强小结

- **Memory** 升一层把 RAG 装进了一个**带状态的对象**——状态就是 history，钩子就是 `ask`。
- **Multi-Doc Agent** 再升一层把 RAG 拆成了**多个对象 + 一个 LLM 路由器**——状态就是路由器选择，钩子还是 `ask`。
- 共用底座：`run_session_eval(system, qna_dict)` + `build_compare_table` 把两者纳入同一对比口径。

下一节"评估"会把这些信号（命中率、对比分、路由正确率）正式纳入工业级评估体系。